# Theorem 10 — sampling-gap shift bound

**Formal source:** [`../10_sampling_gap_shift_bound.md`](../10_sampling_gap_shift_bound.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
p = np.array([0.01, 0.02, 0.05, 0.08, 0.11, 0.14])
q = np.array([0.015, 0.025, 0.055, 0.075, 0.105, 0.16])
pole = -0.4 + 3j
transform_p = np.mean(np.exp(pole * p))
transform_q = np.mean(np.exp(pole * q))
w1 = np.mean(abs(np.sort(p) - np.sort(q)))
lhs, rhs = abs(transform_p - transform_q), abs(pole) * w1
assert lhs <= rhs + 1e-12
print({"transform_shift": float(lhs), "bound": float(rhs), "W1": float(w1)})

In [ ]:
print('THEORY_DEMO_PASS::10_sampling_gap_shift_bound')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')